### Plasticity Data Assembly 2 

- Normalizations
    - **dev data**: raw counts, unnormalized, unlogged
    - **cancer data**: normalized but unlogged (likely library-size normalization or CPM/TPM style), raw data available

The **Masked Language Model (MLM)** pipeline:
- For each cell, pick top-K genes by *relative* expression value → turn those into tokens.
- The only thing that matters for the input is the within-cell ranking of genes, not the absolute units.

In [1]:
import os, re
import pandas as pd
from typing import Tuple, Dict, List, Iterable, Optional

import numpy as np
import scanpy as sc
import anndata as ad
import scipy.sparse as sp

import time
import requests
import gzip

In [2]:
# this auxiliary function checks whether an AnnData object is raw counts or normalized/log-transformed
def check_raw(adata, name="adata"):
    X = adata.X

    # extract only non-zero entries
    if sp.issparse(X):
        vals = X.data
    else:
        arr = np.asarray(X)
        vals = arr[arr > 0]

    print(f"--- {name} ---")
    print("Number of non-zero entries:", vals.size)

    if vals.size == 0:
        print("All zeros → not raw.")
        return

    # integer-ness check
    int_fraction = np.mean(np.isclose(vals, np.round(vals), atol=1e-8))
    print("Fraction of approx-integer entries:", int_fraction)

    print("Min nonzero:", vals.min())
    print("Max:", vals.max())

    if int_fraction > 0.999:
        print("→ Very likely RAW counts.")
        print("")
    else:
        print("→ Normalized or log-transformed.")
        print("")

In [3]:
# .. paths (adjust to your own paths)
commonpath = "/home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/"
DEV_PATH    = os.path.join(commonpath, "development/adata_dev_endoderm_balanced.h5ad")     
CANCER_PATH = os.path.join(commonpath, "pdac/burdziak_epifilterout_kept_N1_N2.h5ad")
OUTDIR      = os.path.join(commonpath, "out/compare_dev_cancer")
os.makedirs(OUTDIR, exist_ok=True)

In [4]:
# .. load to memory 
A_dev    = ad.read_h5ad(DEV_PATH)
A_cancer_original = ad.read_h5ad(CANCER_PATH)

# cancer data must have raw counts stored here
assert A_cancer_original.raw is not None, "Cancer raw counts missing!"

In [5]:
# Convert .raw into a standalone in-memory AnnData
A_cancer = ad.AnnData(
    X   = A_cancer_original.raw.X.copy(),
    obs = A_cancer_original.obs.copy(),
    var = A_cancer_original.raw.var.copy(),
)

A_cancer_original.file.close()

print("Cancer raw matrix:",A_cancer.shape)
print("Developmental (balanced) raw matrix:", A_dev.shape)

Cancer raw matrix: (28131, 16826)
Developmental (balanced) raw matrix: (34760, 24552)


In [6]:
check_raw(A_dev, "development")
check_raw(A_cancer_original, "original cancer")
check_raw(A_cancer, "cancer")

--- development ---
Number of non-zero entries: 59529234
Fraction of approx-integer entries: 1.0
Min nonzero: 1.0
Max: 4669.0
→ Very likely RAW counts.

--- original cancer ---
Number of non-zero entries: 90631118
Fraction of approx-integer entries: 3.582654690412183e-05
Min nonzero: 0.04342219
Max: 6833.936
→ Normalized or log-transformed.

--- cancer ---
Number of non-zero entries: 90631118
Fraction of approx-integer entries: 1.0
Min nonzero: 1.0
Max: 46624.0
→ Very likely RAW counts.



>different symbols! 
- Dev: A_dev.var_names = Ensembl IDs (mm10/GRCm38, GENCODE VM12)
- Cancer: A_cancer.var_names = mm9-era symbols (RIK clones, Gm, RefSeq-style, etc.)


In [7]:
print(A_dev.var_names[:10]),print(A_cancer.var_names[:1000])

Index(['ENSMUSG00000051951', 'ENSMUSG00000102343', 'ENSMUSG00000025900',
       'ENSMUSG00000025902', 'ENSMUSG00000104328', 'ENSMUSG00000033845',
       'ENSMUSG00000025903', 'ENSMUSG00000104217', 'ENSMUSG00000033813',
       'ENSMUSG00000002459'],
      dtype='object')
Index(['0610007P14RIK', '0610009B22RIK', '0610009L18RIK', '0610009O20RIK',
       '0610010F05RIK', '0610010K14RIK', '0610012G03RIK', '0610030E20RIK',
       '0610033M10RIK', '0610037L13RIK',
       ...
       'AGRN', 'AGRP', 'AGT', 'AGTPBP1', 'AGTR1A', 'AGTR2', 'AGTRAP', 'AGXT',
       'AHCTF1', 'AHCY'],
      dtype='object', length=1000)


(None, None)

>#### Stragegy:

- Map cancer symbols → Ensembl gene IDs (as best as possible). Keep dev Ensembl IDs as they are.
- Take the intersection of Ensembl IDs between dev and cancer.

So the overlap is defined in Ensembl ID space, not symbol space.

The correct target for building a shared gene set between the developmental atlas and the PDAC data is the *modern* Ensembl GRCm39 annotation (Ensembl 111).

In [8]:
REF_PATH    = os.path.join(commonpath, "ref/Mus_musculus.GRCm39.111.gtf")
with open(REF_PATH) as f:
    for _ in range(10):
        print(next(f).strip())

#!genome-build GRCm39
#!genome-version GRCm39
#!genome-date 2020-06
#!genome-build-accession GCA_000001635.9
#!genebuild-last-updated 2023-08
1	havana	gene	108344807	108347562	.	+	.	gene_id "ENSMUSG00000104478"; gene_version "2"; gene_name "Gm38212"; gene_source "havana"; gene_biotype "TEC";
1	havana	transcript	108344807	108347562	.	+	.	gene_id "ENSMUSG00000104478"; gene_version "2"; transcript_id "ENSMUST00000194081"; transcript_version "2"; gene_name "Gm38212"; gene_source "havana"; gene_biotype "TEC"; transcript_name "Gm38212-201"; transcript_source "havana"; transcript_biotype "TEC"; tag "basic"; tag "Ensembl_canonical"; transcript_support_level "NA (assigned to previous version 1)";
1	havana	exon	108344807	108347562	.	+	.	gene_id "ENSMUSG00000104478"; gene_version "2"; transcript_id "ENSMUST00000194081"; transcript_version "2"; exon_number "1"; gene_name "Gm38212"; gene_source "havana"; gene_biotype "TEC"; transcript_name "Gm38212-201"; transcript_source "havana"; transcript_bioty

Some helper functions

In [9]:
def strip_ens_version(ids: Iterable[str]) -> pd.Index:
    """
    Strip Ensembl version suffixes: ENSMUSG00000025945.1 -> ENSMUSG00000025945
    Returns a pandas Index of cleaned IDs.
    """
    return pd.Index([g.split('.')[0] for g in ids])

In [10]:
def collapse_duplicate_genes(adata: sc.AnnData, agg: str = "sum") -> sc.AnnData:
    """
    Simplified version: if duplicated var_names exist, keep the first
    occurrence and drop the rest.

    In practice, duplicates after symbol→Ensembl mapping are rare aliases.
    Dropping extra copies is much cheaper and almost never changes biology.

    Parameters
    ----------
    adata - AnnData with possibly duplicated var_names (Ensembl IDs).
    agg - Ignored here, kept only for API compatibility.

    Returns
    -------
    new_adata
        AnnData with unique var_names.
    """
    if not adata.var_names.has_duplicates:
        return adata

    n_dup = adata.var_names.duplicated().sum()
    print(f"[collapse_duplicate_genes] Found {n_dup} duplicate genes. "
          f"Keeping first occurrence and dropping the rest.")

    keep = ~adata.var_names.duplicated(keep="first")
    # Slice columns to keep, copy to detach view
    new_adata = adata[:, keep].copy()
    return new_adata


In [11]:
def parse_gtf_symbol_maps(
    gtf_path: str,
    gene_tag: str = "gene",
    gene_name_key: str = "gene_name"
) -> Tuple[Dict[str, str], Dict[str, str], Dict[str, str]]:
    """
    Parse a GTF file (e.g. Mus_musculus.GRCm39.111.gtf.gz) to build:
      ens2sym:   Ensembl gene ID (no version) -> gene_name
      sym2ens_exact: gene_name (as-is)       -> Ensembl ID
      sym2ens_upper: gene_name.upper()       -> Ensembl ID

    Only 'gene' features are used. If multiple transcripts share a gene_name,
    the first encountered mapping is kept.
    """
    ens2sym: Dict[str, str] = {}
    sym2ens_exact: Dict[str, str] = {}
    sym2ens_upper: Dict[str, str] = {}

    def parse_attr_field(attr_str: str) -> Dict[str, str]:
        d = {}
        for field in attr_str.strip().split(';'):
            field = field.strip()
            if not field:
                continue
            if ' ' not in field:
                continue
            k, v = field.split(' ', 1)
            v = v.strip().strip('"')
            d[k] = v
        return d

    opener = gzip.open if gtf_path.endswith(".gz") else open

    with opener(gtf_path, "rt") as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 9:
                continue
            feature = parts[2]
            if feature != gene_tag:
                continue

            attrs = parse_attr_field(parts[8])
            gene_id = attrs.get("gene_id", None)
            gene_name = attrs.get(gene_name_key, None)
            if gene_id is None or gene_name is None:
                continue

            gene_id = gene_id.split(".")[0]  # strip version
            # Fill ens2sym
            if gene_id not in ens2sym:
                ens2sym[gene_id] = gene_name

            # Fill sym2ens (keep first mapping for a given name)
            if gene_name not in sym2ens_exact:
                sym2ens_exact[gene_name] = gene_id

            up = gene_name.upper()
            if up not in sym2ens_upper:
                sym2ens_upper[up] = gene_id

    print(f"[parse_gtf_symbol_maps] Parsed {len(ens2sym)} genes with gene_name from {gtf_path}")
    return ens2sym, sym2ens_exact, sym2ens_upper


In [12]:
def map_symbols_to_ensembl(
    symbols: Iterable[str],
    sym2ens_upper: Dict[str, str],
    extra_alias: Optional[Dict[str, str]] = None,
) -> Tuple[pd.Series, list]:
    """
    Map a list/Index of gene symbols to Ensembl IDs using:
      1) extra_alias dict (symbol -> Ensembl)
      2) GTF-derived sym2ens_upper (case-insensitive gene_name)
    Returns:
      ens_series: pd.Series of Ensembl IDs (NaN if no match), index = symbols
      not_found:  list of symbols with no mapping
    """
    if extra_alias is None:
        extra_alias = {}

    symbols = pd.Index(symbols)
    ens_ids = []
    not_found = []

    for s in symbols:
        if s in extra_alias:
            ens_ids.append(extra_alias[s])
            continue
        up = s.upper()
        ens = sym2ens_upper.get(up, None)
        if ens is None:
            ens_ids.append(np.nan)
            not_found.append(s)
        else:
            ens_ids.append(ens)

    ens_series = pd.Series(ens_ids, index=symbols, name="ensembl_id")
    return ens_series, not_found

**Optional: external resolver for legacy mm9 symbols**
- Useful for rescuing RIK/Gm/clone IDs that don’t exist as gene_name in the GRCm39 GTF.\
(*If you don’t want live HTTP calls in your pipeline, you can skip this section entirely and use only the GTF-based mapping*)

In [13]:
ENSEMBL_SERVER = "https://rest.ensembl.org"
MYGENE_SERVER = "https://mygene.info/v3"


def query_ensembl_symbol_mouse(symbol: str,
                               max_retries: int = 3,
                               sleep_sec: float = 0.1):
    """
    Query Ensembl REST xrefs/symbol/mus_musculus/{symbol}.
    Returns (ensembl_id, display_name) or (None, None).
    """
    symbol = symbol.strip()
    if not symbol:
        return None, None

    ext = f"/xrefs/symbol/mus_musculus/{symbol}?"
    url = ENSEMBL_SERVER + ext
    headers = {"Content-Type": "application/json"}

    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=headers, timeout=10)
            if not r.ok:
                if r.status_code == 404:
                    return None, None
                time.sleep(sleep_sec)
                continue
            data = r.json()
            if not data:
                return None, None

            # prioritize 'gene' entries
            gene_entries = [d for d in data if d.get("type") == "gene"]
            if not gene_entries:
                gene_entries = data

            entry = gene_entries[0]
            ens_id = entry.get("id")
            disp = entry.get("display_id")
            return ens_id, disp
        except Exception:
            if attempt == max_retries - 1:
                return None, None
            time.sleep(sleep_sec)

    return None, None


def query_mygene_symbol_mouse(symbol: str,
                              max_retries: int = 3,
                              sleep_sec: float = 0.1):
    """
    Query MyGene.info for a mouse symbol and return (ensembl_id, resolved_symbol) or (None, None).
    """
    symbol = symbol.strip()
    if not symbol:
        return None, None

    params = {
        "q": f"symbol:{symbol}",
        "species": "mouse",
        "fields": "ensembl.gene,symbol,name,taxid",
        "size": 5,
    }
    url = f"{MYGENE_SERVER}/query"

    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=10)
            if not r.ok:
                time.sleep(sleep_sec)
                continue
            data = r.json()
            hits = data.get("hits", [])
            if not hits:
                return None, None
            hits = sorted(hits, key=lambda h: h.get("_score", 0), reverse=True)
            for h in hits:
                ens = h.get("ensembl")
                if not ens:
                    continue
                if isinstance(ens, list):
                    ens_id = ens[0].get("gene")
                else:
                    ens_id = ens.get("gene")
                if ens_id:
                    sym = h.get("symbol") or symbol
                    return ens_id, sym
            return None, None
        except Exception:
            if attempt == max_retries - 1:
                return None, None
            time.sleep(sleep_sec)

    return None, None


def resolve_mouse_symbols(symbols: Iterable[str],
                          sleep_sec: float = 0.15):
    """
    For a list of mouse gene symbols (e.g. mm9-era RIK/GM/clone IDs), try:
      1) Ensembl REST xrefs/symbol
      2) MyGene.info
    Returns:
      df: DataFrame with columns:
           original_symbol, ensembl_id, resolved_symbol, source
      manual_alias: dict {original_symbol -> ensembl_id} for successfully mapped symbols
    """
    records = []
    symbols = list(symbols)
    for i, s in enumerate(symbols):
        s = str(s).strip()
        if not s:
            continue

        ens_id, rsym = query_ensembl_symbol_mouse(s)
        source = None
        if ens_id is not None:
            source = "ensembl"
        else:
            ens_id, rsym = query_mygene_symbol_mouse(s)
            if ens_id is not None:
                source = "mygene"

        records.append(
            {
                "original_symbol": s,
                "ensembl_id": ens_id,
                "resolved_symbol": rsym,
                "source": source,
            }
        )

        if (i + 1) % 20 == 0:
            print(f"[resolve_mouse_symbols] Processed {i + 1}/{len(symbols)} symbols...")
        time.sleep(sleep_sec)

    df = pd.DataFrame(records)
    manual_alias = {
        row["original_symbol"]: row["ensembl_id"]
        for _, row in df.dropna(subset=["ensembl_id"]).iterrows()
    }

    print(
        f"[resolve_mouse_symbols] Resolved {len(manual_alias)}/{len(symbols)} symbols "
        f"({len(symbols) - len(manual_alias)} with no hit)."
    )

    return df, manual_alias


**Main pipeline**: translate + align

In [14]:
def translate_and_align_to_ensembl(
    A_dev: sc.AnnData,
    A_cancer: sc.AnnData,
    gtf_path: str,
    duplicate_agg: str = "sum",
    manual_alias: Optional[Dict[str, str]] = None,
    use_external_resolver: bool = False,
    external_resolve_min_detect_frac: float = 0.0,
) -> Tuple[sc.AnnData, sc.AnnData, dict]:
    """
    Normalize A_dev (Ensembl IDs) and A_cancer (symbols) into Ensembl IDs in the
    GRCm39 / Ensembl 111 namespace and align them to their intersection.

    Parameters
    ----------
    A_dev
        Development dataset. Assumed to have Ensembl IDs in .var_names
        (may have version suffixes).
    A_cancer
        PDAC dataset. Assumed to have *symbols* in .var_names (mm9-era style).
    gtf_path
        Path to Mus_musculus.GRCm39.111.gtf(.gz).
    duplicate_agg
        How to collapse duplicate Ensembl IDs in A_cancer after mapping: 'sum' or 'mean'.
    manual_alias
        Optional dict {symbol -> Ensembl ID} with hand-curated mappings
        (e.g. RIK clones, Gm genes).
    use_external_resolver
        If True, use Ensembl REST + MyGene to try to rescue unmapped symbols.
    external_resolve_min_detect_frac
        If >0, only symbols with detection_fraction >= this threshold in A_cancer
        will be sent to the external resolver (to save time / avoid junk).

    Returns
    -------
    A_dev_aln, A_cancer_aln, summary
        Aligned AnnData objects and a summary dict of mapping/overlap statistics.
    """
    if manual_alias is None:
        manual_alias = {}

    # --- build GTF-based maps ---
    ens2sym, sym2ens_exact, sym2ens_upper = parse_gtf_symbol_maps(gtf_path)

    # --- process dev (already Ensembl IDs) ---
    A_dev = A_dev.copy()
    A_dev.var_names = strip_ens_version(A_dev.var_names)
    A_dev.var["symbol"] = [ens2sym.get(g, g) for g in A_dev.var_names]

    # --- process cancer: symbols -> Ensembl ---
    A_cancer = A_cancer.copy()
    orig_symbols = A_cancer.var_names.copy()

    # 1) initial GTF-based mapping
    ens_names, not_found = map_symbols_to_ensembl(orig_symbols, sym2ens_upper, extra_alias=manual_alias)

    n_total = len(orig_symbols)
    mask_mapped_gtf = ens_names.notna()
    n_mapped_gtf = int(mask_mapped_gtf.sum())
    n_unmapped_gtf = int((~mask_mapped_gtf).sum())

    print(f"[map] GTF-based mapping: {n_mapped_gtf}/{n_total} symbols mapped, {n_unmapped_gtf} unmapped")

    # 2) optional external resolver for unmapped symbols
    manual_alias_ext = {}
    if use_external_resolver and n_unmapped_gtf > 0:
        unmapped_syms = orig_symbols[~mask_mapped_gtf]

        if external_resolve_min_detect_frac > 0:
            # compute detection fraction to restrict external calls to "important" genes
            mask_unmapped_in_cancer = A_cancer.var_names.isin(unmapped_syms)
            X = A_cancer[:, mask_unmapped_in_cancer].X
            if not isinstance(X, np.ndarray):
                X = X.toarray()
            det_frac = (X > 0).mean(axis=0)
            df_unmapped = pd.DataFrame({
                "gene": A_cancer.var_names[mask_unmapped_in_cancer].tolist(),
                "detection_fraction": det_frac,
            })
            # filter
            keep = df_unmapped["detection_fraction"] >= external_resolve_min_detect_frac
            syms_to_resolve = df_unmapped.loc[keep, "gene"].tolist()
            print(f"[external] Resolving {len(syms_to_resolve)} high-detection unmapped symbols externally "
                  f"(threshold={external_resolve_min_detect_frac}).")
        else:
            syms_to_resolve = list(unmapped_syms)
            print(f"[external] Resolving all {len(syms_to_resolve)} unmapped symbols externally.")

        if len(syms_to_resolve) > 0:
            df_ext, manual_alias_ext = resolve_mouse_symbols(syms_to_resolve)
            print(df_ext.head())
        else:
            print("[external] No symbols passed the detection threshold; skipping external resolver.")

        # merge alias maps
        manual_alias_combined = dict(manual_alias)
        manual_alias_combined.update(manual_alias_ext)

        # re-run full mapping with enriched alias dict
        ens_names, not_found = map_symbols_to_ensembl(orig_symbols, sym2ens_upper, extra_alias=manual_alias_combined)
        mask_mapped_final = ens_names.notna()
        n_mapped_final = int(mask_mapped_final.sum())
        print(f"[map] After external rescue: {n_mapped_final}/{n_total} symbols mapped.")
    else:
        manual_alias_combined = dict(manual_alias)
        mask_mapped_final = ens_names.notna()
        n_mapped_final = int(mask_mapped_final.sum())

    # final unmapped set
    n_unmapped_final = int((~mask_mapped_final).sum())
    unmapped_final = orig_symbols[~mask_mapped_final]
    if n_unmapped_final:
        print(f"[map] Final unmapped symbols: {n_unmapped_final}. "
              f"Examples: {list(unmapped_final[:20])}")

    # --- apply mapping to A_cancer ---
    kept_symbols = orig_symbols[mask_mapped_final]
    kept_ens = ens_names[mask_mapped_final]

    A_cancer = A_cancer[:, mask_mapped_final].copy()
    A_cancer.var_names = kept_ens.values
    A_cancer.var["symbol_orig"] = kept_symbols
    A_cancer.var["symbol"] = [ens2sym.get(g, g) for g in A_cancer.var_names]

    # --- collapse duplicates after mapping (two symbols -> same Ensembl ID) ---
    if A_cancer.var_names.has_duplicates:
        A_cancer = collapse_duplicate_genes(A_cancer, agg=duplicate_agg)

    # --- compute intersection in Ensembl ID space ---
    common = A_dev.var_names.intersection(A_cancer.var_names)

    # Convert to a sorted Index or list to ensure stable ordering
    common = pd.Index(common)
    common = common.sort_values()

    print(
        f"[align] A_dev genes: {A_dev.n_vars}, "
        f"A_cancer mapped genes: {A_cancer.n_vars}, "
        f"common Ensembl IDs: {len(common)}"
    )

    if len(common) == 0:
        raise ValueError(
            "No common Ensembl IDs between dev and cancer after mapping. "
            "Check that both are mapped into the same Ensembl namespace."
        )

    # Use .loc[:, common] which is clearer and sometimes safer
    A_dev_aln = A_dev[:, common].copy()
    A_cancer_aln = A_cancer[:, common].copy()

    # Attach symbols
    A_dev_aln.var["symbol"] = [ens2sym.get(g, g) for g in common]
    A_cancer_aln.var["symbol"] = A_dev_aln.var["symbol"].values

    summary = {
        "gtf_path": gtf_path,
        "A_dev_vars": int(A_dev.n_vars),
        "A_cancer_vars_original": int(len(orig_symbols)),
        "A_cancer_vars_mapped_final": int(A_cancer.n_vars),
        "mapped_by_gtf": n_mapped_gtf,
        "mapped_after_external": n_mapped_final,
        "unmapped_final": n_unmapped_final,
        "common_vars": int(len(common)),
        "n_manual_alias_input": len(manual_alias),
        "n_manual_alias_external": len(manual_alias_ext) if use_external_resolver else 0,
        "unmapped_examples": list(unmapped_final[:50]),
    }

    return A_dev_aln, A_cancer_aln, summary


In [15]:
A_dev_aln, A_cancer_aln, summary = translate_and_align_to_ensembl(
    A_dev,
    A_cancer,
    gtf_path=REF_PATH,
    duplicate_agg="sum",
    manual_alias= None,              # or your curated dict
    use_external_resolver=True,     # optional, slower
    external_resolve_min_detect_frac=0.2,  # e.g. only rescue well-expressed genes
)

print(summary)

[parse_gtf_symbol_maps] Parsed 56655 genes with gene_name from /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/ref/Mus_musculus.GRCm39.111.gtf
[map] GTF-based mapping: 15676/16826 symbols mapped, 1150 unmapped
[external] Resolving 278 high-detection unmapped symbols externally (threshold=0.2).
[resolve_mouse_symbols] Processed 20/278 symbols...
[resolve_mouse_symbols] Processed 40/278 symbols...
[resolve_mouse_symbols] Processed 60/278 symbols...
[resolve_mouse_symbols] Processed 80/278 symbols...
[resolve_mouse_symbols] Processed 100/278 symbols...
[resolve_mouse_symbols] Processed 120/278 symbols...
[resolve_mouse_symbols] Processed 140/278 symbols...
[resolve_mouse_symbols] Processed 160/278 symbols...
[resolve_mouse_symbols] Processed 180/278 symbols...
[resolve_mouse_symbols] Processed 200/278 symbols...
[resolve_mouse_symbols] Processed 220/278 symbols...
[resolve_mouse_symbols] Processed 240/278 symbols...
[resolve_mouse_symbols] Processed 260/278 symbols...
[resolve_m

In [16]:
# Sanity checks: make sure the two objects really align on the same genes (order matters) ---
assert (A_dev_aln.var_names == A_cancer_aln.var_names).all(), "Gene orders differ"
assert (A_dev_aln.var["symbol"].values == A_cancer_aln.var["symbol"].values).all(), "Gene symbols differ"

In [ ]:
# one can check these examples ones here: https://mart.ensembl.org/index.html
for gene in ["Krt8", "Krt18", "Epcam", "Muc1", "Krt19", "Sox9", "Mki67"]:
    hits = A_cancer_aln.var.query("symbol == @gene").index
    print(gene, "->", list(hits))


Krt8 -> ['ENSMUSG00000049382']
Krt18 -> ['ENSMUSG00000023043']
Epcam -> ['ENSMUSG00000045394']
Muc1 -> ['ENSMUSG00000042784']
Krt19 -> ['ENSMUSG00000020911']
Sox9 -> ['ENSMUSG00000000567']
Mki67 -> ['ENSMUSG00000031004']


In [18]:
print('dev data: ',A_dev_aln.shape)
print('cancer data: ', A_cancer_aln.shape)

dev data:  (34760, 15376)
cancer data:  (28131, 15376)


In [19]:
check_raw(A_dev_aln, "development")
check_raw(A_cancer_aln, "cancer")

--- development ---
Number of non-zero entries: 57139701
Fraction of approx-integer entries: 1.0
Min nonzero: 1.0
Max: 1803.0
→ Very likely RAW counts.

--- cancer ---
Number of non-zero entries: 87098503
Fraction of approx-integer entries: 1.0
Min nonzero: 1.0
Max: 46624.0
→ Very likely RAW counts.



In [26]:
print("DEV obs columns:", A_dev_aln.obs.columns)
print('')
print("CANCER obs columns:", A_cancer_aln.obs.columns)

DEV obs columns: Index(['cell_id', 'keep', 'day', 'embryo_id', 'experimental_batch',
       'cell_id_base', 'major_trajectory', 'celltype_update', 'endo_category',
       'day_meta', 'dev_state_pancreas', 'dev_state_union', 'dev_day',
       'source_file', 'dev_tier'],
      dtype='object')

CANCER obs columns: Index(['batch', 'condition', 'cluster', 'epi_score', 'cancer_state',
       'condition_collapsed'],
      dtype='object')


In [28]:
# .. save
out_dev   = os.path.join(commonpath, "development/A_dev_aligned.h5ad")
out_cancer = os.path.join(commonpath, "pdac/A_cancer_aligned.h5ad")

A_dev_aln.write(out_dev)
A_cancer_aln.write(out_cancer)

print("Saved:\n ", out_dev, "\n ", out_cancer)


Saved:
  /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/development/A_dev_aligned.h5ad 
  /home/jfpoyatos/Work/MNCN_24_25/Development_and_PDAC/data/pdac/A_cancer_aligned.h5ad
